In [1]:
import boto3
import json
from IPython.display import JSON

In [2]:
bedrock = boto3.client(service_name="bedrock-runtime", region_name="us-east-1")
bedrock_agent = boto3.client(service_name="bedrock-agent", region_name="us-east-1")

In [18]:
MODEL_ID_NOVA_PRO = "amazon.nova-pro-v1:0"
MODEL_ID_NOVA_LITE = "amazon.nova-lite-v1:0"
MODEL_ID_LLAMA_70B = "meta.llama3-1-70b-instruct-v1:0"
MODEL_ID_GPT_OSS_120B= "openai.gpt-oss-120b-1:0"

In [4]:
from nt import system


temperature = 0.7
#top_k = 200

inference_config = {"temperature": temperature}

#additional_model_fields = {"top_k": top_k}

system_prompts = [{
    "text": "Você é um app que cria playlists para uma rádio que toca rock e pop. " + "Retorne apenas o nome das músicas e o artista"
}]

messages = []

message_1 = {
    "role": "user",
    "content": [{
        "text": "Crie uma lista com 3 músicas de rock"
    }]
}

messages.append(message_1)

response = bedrock.converse(
    modelId=MODEL_ID_NOVA_PRO,
    messages=messages,
    system=system_prompts,
    inferenceConfig=inference_config,
    #additionalModelRequestFields=additional_model_fields
)

In [ ]:
def print_response(response):
    model_response = response.get('output', {}).get('message', {}).get('content', {})[0].get('text', '')

    print("Resposta do modelo:")
    print("Músicas pop sugeridas")
    print(model_response)

print_response(response)

Resposta do modelo:
Músicas pop sugeridas
Claro! Aqui estão três músicas de rock:

1. "Bohemian Rhapsody" - Queen
2. "Sweet Child o' Mine" - Guns N' Roses
3. "Smells Like Teen Spirit" - Nirvana


In [6]:
message_2 = {
    "role": "user",
    "content": [{"text": "Garanta que as músicas sejam de artistas do Reino Unido"}]
}

messages.append(message_2)

In [31]:
""" NOVA LITE """

response = bedrock.converse(
    modelId=MODEL_ID_NOVA_LITE,
    messages=messages,
    system=system_prompts,
    inferenceConfig=inference_config,
    #additionalModelRequestFields=additional_model_fields
)

print_response(response)

Resposta do modelo:
Músicas pop sugeridas
"Sweet Child O' Mine" - Guns N' Roses

"Back in Black" - AC/DC

"Bohemian Rhapsody" - Queen


In [ ]:
""" GPT OSS 120B """

response = bedrock.converse(
    modelId=MODEL_ID_GPT_OSS_120B,
    messages=messages,
    system=system_prompts,
    inferenceConfig=inference_config,
    #additionalModelRequestFields=additional_model_fields
)

response['output']['message']['content'][1]['text']

'Bohemian Rhapsody – Queen  \nParanoid – Black Sabbath  \nLondon Calling – The Clash'

## Converse com prompt

In [33]:
try:
    response = bedrock_agent.create_prompt(
        name="Event-Booking-Screener-1",
        description="Verifica se um pedido de reserva atende aos requisitos do espaço",
        variants=[
            {
                "name": "Variant1",
                "modelId": MODEL_ID_NOVA_PRO,
                "templateType": "CHAT",
                "inferenceConfiguration": {
                    "text": {
                        "temperature": 0.4
                    }
                },
                "templateConfiguration": {
                    "chat": {
                        'system': [
                            {
                                "text": """Você está ajudando o gerente de um espaço de eventos a fazer uma triagem rápida dos pedidos de reserva.
Leia a mensagem inteira e responda SIM ou NÃO para cada item abaixo.
Seja rigoroso e, se não tiver informação suficiente, responda NÃO.

A) O pedido menciona menos de 150 convidados?
B) O horário solicitado está entre 10h e 23h?
C) O evento evita atividades proibidas, como animais vivos ou fogo aberto?
D) Há alguma indicação de que o serviço de bebidas alcoólicas será feito por um fornecedor licenciado?
E) Há informação suficiente para dar retorno a quem fez o pedido?"""
                            }
                        ],
                        'messages': [{
                            'role': 'user',
                            'content': [
                                {
                                    'text': "Pedido de reserva: {{event_request}}"
                                }
                            ]
                        }],
                        'inputVariables': [
                            { 'name': 'event_request' }
                        ]
                    }
                }
            }
        ]
    )
    print("Criado!")
    prompt_arn = response.get("arn")
except bedrock.exceptions.ConflictException as e:
    print("Já existe!")
    response = bedrock.list_prompts()
    prompt = next((prompt for prompt in response['promptSummaries'] if prompt['name'] == "EventBookingScreener_xyz"), None)
    prompt_arn = prompt['arn']

prompt_arn

Criado!


'arn:aws:bedrock:us-east-1:<ACCOUNT_ID>:prompt/WRXISW2W3X'

In [13]:
response = bedrock.converse(
    modelId=prompt_arn,
    promptVariables={
        'event_request': {
            'text': """
                Olá, gostaria de reservar o espaço de vocês para uma festa de aniversário. Esperamos cerca de 90 convidados,
                com início por volta das 18h e término até as 22h30. Teremos apenas
                um DJ e pista de dança. Queremos servir vinho e cerveja - meu primo tem uma empresa de buffet com licença
                para venda de bebidas alcoólicas que cuidaria disso. Me avisem se a data está disponível e quais são os próximos passos. Obrigado!
            """
        }
    },
)
print(response['output']['message']['content'][0]['text'])

A) YES - The request mentions around 90 guests, which is fewer than 150.
B) YES - The requested time is from 6pm to 10:30pm, which is within the 10am to 11pm window.
C) YES - The event description (DJ and dancing) does not mention restricted activities like live animals or open flames.
D) YES - The message indicates that a licensed catering company will handle alcohol service.
E) YES - There is enough information provided (guest count, time, type of event, alcohol service) to follow up with the requester.
